In [ ]:
from gdrive_fsspec import GoogleDriveFileSystem
import marimo as mo

fs = GoogleDriveFileSystem(use_listings_cache=False, skip_instance_cache=True, auth_kwargs={"use_local_webserver": False})
mo.output.clear_console()

In [ ]:
fs.get("Data_Science_Project/checkpoints/simple_enhancer_v5/simple_enhancer_best.pt", "marimo")
fs.get("Data_Science_Project/checkpoints/sa_fusion_v6_final/simple_fusion_best.pt", "/marimo/")

In [ ]:
import os
from pathlib import Path
from fsspec.callbacks import TqdmCallback
from concurrent.futures import ThreadPoolExecutor

# 1. Define paths and roots
DATA_ROOT = Path("/marimo/data")
VOX_DIR = DATA_ROOT / "voxceleb1"
MUSAN_DIR = DATA_ROOT / "musan"
VOX_TEST_DIR = DATA_ROOT / "voxceleb1_test"

# Setup target folders safely
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# 2. Execution Targets
datasets_to_download = [
    {"remote_path": "Data_Science_Project/data/vox1_dev_wav.zip", "extract_to": VOX_DIR, "name": "voxceleb1", "extension": ".zip"},
    {"remote_path": "Data_Science_Project/data/musan.tar.gz", "extract_to": MUSAN_DIR, "name": "musan", "extension": ".tar.gz"},
    {"remote_path": "Data_Science_Project/data/vox1_test_wav.zip", "extract_to": VOX_TEST_DIR, "name": "voxceleb1_test", "extension": ".zip"}
]

def download_and_extract_worker(task):
    """Worker thread with a thread-isolated filesystem client"""
    remote_path = task["remote_path"]
    extract_to_path = Path(task["extract_to"])
    name = task["name"]
    extension = task["extension"]
    local_archive = Path(f"/marimo/{name}{extension}")

    # Short-circuit: skip if folder exists and already contains data
    if extract_to_path.exists() and any(extract_to_path.iterdir()):
        print(f"✓ {name} already exists on disk. Skipping download.")
        return

    # CRITICAL: Create an isolated Google Drive client specifically for THIS thread
    # Replace GDriveFileSystem() with the exact way you instantiated 'fs' originally
    from gdrive_fsspec import GoogleDriveFileSystem
    thread_safe_fs = GoogleDriveFileSystem() 

    # 1. Thread-Isolated Download Stream
    print(f"📥 Starting parallel download for {name}...")
    tqdm_settings = {"desc": f"Downloading {name}", "unit": "B", "unit_scale": True, "unit_divisor": 1024, "leave": False}

    thread_safe_fs.get(remote_path, str(local_archive), callback=TqdmCallback(tqdm_kwargs=tqdm_settings))

    print(f"🎉 {name} extraction complete!")

# ============================================================
# PARALLEL EXECUTION MATRIX
# ============================================================
print("🚀 Initiating thread-isolated parallel download streams...")

with ThreadPoolExecutor(max_workers=3) as executor:
    list(executor.map(download_and_extract_worker, datasets_to_download))

print("\n✅ All operations completed successfully!")

In [ ]:
def train_and_evaluate_fusions():
    # /// script
    # requires-python = ">=3.11,<3.14"
    # dependencies = [
    #     "torch", "torchaudio", "transformers",
    #     "denoisers", "speechbrain",
    #     "scikit-learn", "tqdm", "pandas",
    #     "numpy", "soundfile", "huggingface_hub",
    # ]
    # ///
    #
    # ============================================================================
    # TRAIN & COMPARE FUSION ARCHITECTURES ON FROZEN ENHANCER
    # ============================================================================
    # Loads the fine-tuned Wave-U-Net from v6 checkpoint. Trains three fusion
    # models (MLP, Self-Attention, Cross-Attention) with identical training
    # protocol (triplet + preserve, balanced noise sampling). Evaluates all
    # on full VoxCeleb1 test × 3 noise types × 9 SNRs.
    #
    # Outputs:
    #   - Per-condition EER for: raw, enh, MLP, SA, CA
    #   - Summary table by noise type
    #   - Best fusion model identified
    # ============================================================================

    import os
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

    import sys, types, re, json, random, math, enum, gc, time, urllib.request, multiprocessing, hashlib
    import importlib, importlib.machinery, importlib.util
    from pathlib import Path
    from collections import defaultdict, OrderedDict

    # ---- Drive ----
    fs = globals().get("fs", None)
    _DRIVE_ENABLED = fs is not None and os.environ.get("DRIVE_ENABLED", "true").lower() in {"1", "true", "yes"}
    _DRIVE_REMOTE = "Data_Science_Project/checkpoints/fusion_comparison_v1"
    if _DRIVE_ENABLED:
        try:
            fs.makedirs(_DRIVE_REMOTE, exist_ok=True)
            print(f"[drive] ready: {_DRIVE_REMOTE}")
        except Exception as e:
            print(f"[drive] mkdir failed: {e}")

    # ---- torchvision stub ----
    os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"
    os.environ["USE_TORCHVISION"] = "0"
    os.environ["TRANSFORMERS_NO_TORCHVISION_IMAGE"] = "1"
    _BROKEN_TV = True
    try:
        import torchvision  # noqa
        from torchvision.transforms import InterpolationMode  # noqa
        _BROKEN_TV = False
    except Exception:
        pass

    def _install_stub(name, is_package=True):
        mod = types.ModuleType(name)
        spec = importlib.machinery.ModuleSpec(name=name, loader=None, is_package=is_package)
        if is_package:
            spec.submodule_search_locations = []
        mod.__spec__ = spec
        mod.__loader__ = None
        mod.__package__ = name if is_package else name.rpartition(".")[0]
        mod.__path__ = [] if is_package else None
        mod.__file__ = f"<stub:{name}>"
        mod.__version__ = "0.0.0-stub"
        sys.modules[name] = mod
        return mod

    if _BROKEN_TV:
        fake = _install_stub("torchvision", True)
        tr = _install_stub("torchvision.transforms", True)

        class InterpolationMode(enum.Enum):
            NEAREST = "nearest"; NEAREST_EXACT = "nearest-exact"; BILINEAR = "bilinear"
            BICUBIC = "bicubic"; BOX = "box"; HAMMING = "hamming"; LANCZOS = "lanczos"

        tr.InterpolationMode = InterpolationMode
        for c in ("Compose", "Resize", "CenterCrop", "RandomCrop", "Normalize", "ToTensor",
                  "PILToTensor", "ConvertImageDtype", "RandomResizedCrop", "RandomHorizontalFlip",
                  "ColorJitter", "Grayscale", "RandomApply", "Lambda", "ToPILImage"):
            setattr(tr, c, type(c, (), {}))
        tr_func = _install_stub("torchvision.transforms.functional", False)
        for fn in ("resize","center_crop","normalize","to_tensor","pil_to_tensor","to_pil_image",
                   "rgb_to_grayscale","crop","pad","hflip","vflip","rotate","adjust_brightness",
                   "adjust_contrast","adjust_saturation","adjust_hue","convert_image_dtype","gaussian_blur"):
            setattr(tr_func, fn, lambda *a, **k: None)
        tr.functional = tr_func
        tr_v2 = _install_stub("torchvision.transforms.v2", True)
        tr_v2.InterpolationMode = InterpolationMode
        for c in ("Compose","Transform","ToDtype","ToImage","Normalize","Resize","CenterCrop","RandomCrop"):
            setattr(tr_v2, c, type(c, (), {}))
        tr_v2_func = _install_stub("torchvision.transforms.v2.functional", False)
        tr_v2.functional = tr_v2_func; tr.v2 = tr_v2; fake.transforms = tr
        io_mod = _install_stub("torchvision.io", False)
        io_mod.ImageReadMode = type("ImageReadMode", (), {"UNCHANGED":0,"GRAY":1,"GRAY_ALPHA":2,"RGB":3,"RGBA":4})
        for fn in ("decode_image","read_image","write_image","encode_jpeg","encode_png","decode_jpeg","decode_png"):
            setattr(io_mod, fn, lambda *a, **k: None)
        fake.io = io_mod
        fake.models = _install_stub("torchvision.models", True)
        ops = _install_stub("torchvision.ops", False)
        for fn in ("nms","batched_nms","nms_rotated","box_convert","box_iou","generalized_box_iou",
                   "roi_align","roi_pool","ps_roi_align","ps_roi_pool","sigmoid_focal_loss","deform_conv2d"):
            setattr(ops, fn, lambda *a, **k: None)
        for c in ("RoIAlign","RoIPool","PSRoIAlign","PSRoIPool","DeformConv2d"):
            setattr(ops, c, type(c, (), {}))
        fake.ops = ops
        tv = _install_stub("torchvision.tv_tensors", False)
        for c in ("TVTensor","Image","BoundingBoxes","Mask","Video","BoundingBoxFormat"):
            setattr(tv, c, type(c, (), {}))
        fake.tv_tensors = tv
        utils = _install_stub("torchvision.utils", False)
        for fn in ("make_grid","save_image","draw_bounding_boxes","draw_segmentation_masks"):
            setattr(utils, fn, lambda *a, **k: None)
        fake.utils = utils
        print("[stub] torchvision ok")

    _MP_CTX = None
    if os.name == "posix":
        try:
            _MP_CTX = multiprocessing.get_context("fork")
            print("Using fork multiprocessing context")
        except Exception:
            _MP_CTX = None

    import torch.nn as nn
    _orig_getattr = nn.Module.__getattr__
    nn.Module.__getattr__ = lambda self, name: {} if name == "all_tied_weights_keys" else _orig_getattr(self, name)

    import numpy as np, pandas as pd, torch, torch.nn.functional as F, torchaudio
    from torch.utils.data import Dataset, Sampler, DataLoader
    from tqdm import tqdm
    from sklearn.metrics import roc_curve
    from huggingface_hub import hf_hub_download
    from denoisers import WaveUNetModel, WaveUNetConfig

    torch.backends.cudnn.benchmark = False

    def make_autocast(en):
        try: return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=en)
        except (TypeError, AttributeError): return torch.cuda.amp.autocast(enabled=en)
    def make_scaler(en):
        try: return torch.amp.GradScaler("cuda", enabled=en)
        except (TypeError, AttributeError): return torch.cuda.amp.GradScaler(enabled=en)
    def sseed(*p):
        t = "||".join(str(x) for x in p)
        return int.from_bytes(hashlib.sha256(t.encode()).digest()[:8], "little")

    # =========================================================================
    # CONFIG
    # =========================================================================
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    USE_CUDA = DEVICE.startswith("cuda")
    SB_DEVICE = f"cuda:{torch.cuda.current_device()}" if USE_CUDA else "cpu"
    print(f"Device: {DEVICE}")
    if USE_CUDA:
        props = torch.cuda.get_device_properties(0)
        print(f"GPU: {props.name} | VRAM: {props.total_memory/1e9:.1f} GB")
        CHUNK_BS = 32 if props.total_memory / 1e9 >= 40 else (16 if props.total_memory / 1e9 >= 20 else 8)
    else:
        CHUNK_BS = 4

    DEV_ROOT = "/marimo/voxceleb1_dev"
    TEST_ROOT = "/marimo/voxceleb1_test/wav"
    TRIAL_FILE = "/marimo/veri_test.txt"
    MUSAN_ROOT = "/marimo/musan"
    ECAPA_DIR = "/marimo/ecapa_voxceleb"

    # Frozen fine-tuned Wave-U-Net (v6 checkpoint)
    ENH_CKPT = "/marimo/simple_enhancer_best.pt"
    FUS_CKPT_DIR = "/marimo/fusion_comparison_checkpoints"
    os.makedirs(FUS_CKPT_DIR, exist_ok=True)

    SR = 16000; MIN_SEC = 4.0; MAX_SEC = 10.0
    SNR_RANGE = (-20.0, 20.0)
    CLEAN_FRAC = 0.15; CLEAN_SNR = 30.0
    P_SPEAKERS = 12; K_UTTS = 4

    # ---- Fusion training config (identical across all 3 architectures) ----
    FUS_EPOCHS = 30
    FUS_STEPS_PER_EP = 300
    FUS_LR = 5e-4
    FUS_GRAD_CLIP = 5.0
    FUS_WARMUP_STEPS = 150
    TRIPLET_MARGIN = 0.25
    LAMBDA_FUS_PRESERVE = 0.10
    FUS_PRESERVE_SNR_THRESHOLD = 15.0
    FUS_PRESERVE_SNR_TEMP = 3.0

    SA_NUM_HEADS = 4; SA_NUM_LAYERS = 2; SA_DROPOUT = 0.1
    CA_NUM_HEADS = 4; CA_DROPOUT = 0.1

    N_VAL_SPEAKERS = 40; N_VAL_UTTS_PER_SPK = 4
    EVAL_BATCH = 8
    FUS_VAL_SNRS = [-20.0, -15.0, -10.0, -5.0, 0.0, 5.0, 10.0, 20.0]

    NOISE_SUBDIRS = [("babble", "speech"), ("music", "music"), ("noise", "noise")]
    MUSAN_TEST_RATIO = 0.30; MUSAN_SPLIT_SEED_BASE = 1234
    EVAL_SNRS = [-20, -15, -10, -5, 0, 5, 10, 15, 20]
    PAPER_SNRS = [0, -5, -10, -15, -20]

    # =========================================================================
    # SHARED UTILS
    # =========================================================================
    AUDIO_EXTS = {".wav", ".flac", ".ogg", ".mp3"}
    _AC = OrderedDict()
    def _cp(k, v):
        if k in _AC: _AC.move_to_end(k); return
        _AC[k] = v
        while len(_AC) > 5000: _AC.popitem(last=False)
    def list_audio(root):
        if not os.path.isdir(root): return []
        return sorted(str(p) for p in Path(root).rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_EXTS)
    def extract_spk(p):
        for part in Path(p).parts:
            if part.startswith("id") and len(part) >= 5: return part
        return Path(p).parent.name
    def load_wav(p, sr_t=SR):
        k = (str(p), sr_t)
        v = _AC.get(k)
        if v is not None: _AC.move_to_end(k); return v
        try: wav, sr = torchaudio.load(str(p), backend="soundfile")
        except Exception: wav, sr = torchaudio.load(str(p))
        if sr != sr_t: wav = torchaudio.functional.resample(wav, sr, sr_t)
        if wav.dim() == 2: wav = wav.mean(dim=0)
        wav = wav.float().contiguous(); _cp(k, wav); return wav
    def rms(x, eps=1e-8): return torch.sqrt(torch.mean(x ** 2) + eps)
    def mix_at_snr(c, n, snr):
        if n.numel() < c.numel(): n = n.repeat(int(np.ceil(c.numel() / n.numel())))
        if n.numel() > c.numel():
            s = random.randint(0, n.numel() - c.numel()); n = n[s:s + c.numel()]
        sc = rms(c) / (rms(n) * (10 ** (snr / 20)) + 1e-8)
        nc = sc * n; no = c + nc
        pk = torch.maximum(c.abs().max(), no.abs().max()).clamp_min(1e-6)
        return no / pk, c / pk, nc / pk
    def mix_det(c, n, snr, seed_key):
        r = random.Random(sseed(*seed_key))
        if n.numel() < c.numel(): n = n.repeat(int(np.ceil(c.numel() / n.numel())))
        if n.numel() > c.numel():
            s = r.randint(0, n.numel() - c.numel()); n = n[s:s + c.numel()]
        else:
            n = n[:c.numel()]
        sc = rms(c) / (rms(n) * (10 ** (snr / 20)) + 1e-8)
        nc = sc * n; no = c + nc
        pk = torch.maximum(c.abs().max(), no.abs().max()).clamp_min(1e-6)
        return no / pk, c / pk, nc / pk
    def compute_eer(sc, lb):
        s = np.asarray(sc, dtype=np.float64); l = np.asarray(lb, dtype=np.int64)
        if len(np.unique(l)) < 2: return 50.0
        fpr, tpr, _ = roc_curve(l, s); fnr = 1 - tpr
        i = np.nanargmin(np.abs(fnr - fpr)); return float((fpr[i] + fnr[i]) / 2 * 100)
    def split_strat(files, r=0.30, seed=42):
        files = sorted(files); rng = random.Random(seed)
        sh = files.copy(); rng.shuffle(sh)
        n = max(1, int(len(sh) * r)); return sh[n:], sh[:n]
    def _unwrap(o):
        if hasattr(o, "audio"): return o.audio
        if hasattr(o, "waveform"): return o.waveform
        if isinstance(o, (list, tuple)): return o[0]
        if isinstance(o, dict): return next(v for v in o.values() if torch.is_tensor(v))
        return o

    # ---- MUSAN split ----
    print("\nMUSAN split:")
    noise_train_banks = {}; noise_test_banks = {}
    for i, (nt, sub) in enumerate(NOISE_SUBDIRS):
        files = list_audio(os.path.join(MUSAN_ROOT, sub))
        tr, te = split_strat(files, MUSAN_TEST_RATIO, MUSAN_SPLIT_SEED_BASE + i)
        assert not (set(tr) & set(te))
        noise_train_banks[nt] = tr; noise_test_banks[nt] = te
        print(f"  {nt:8s}: train={len(tr):5d}  test={len(te):5d}")

    # ---- ECAPA ----
    print("\nLoading ECAPA ...")
    from speechbrain.inference.speaker import EncoderClassifier
    ecapa = EncoderClassifier.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb",
                                            savedir=ECAPA_DIR, run_opts={"device": SB_DEVICE})
    for p in ecapa.parameters(): p.requires_grad_(False)
    ecapa.eval()

    @torch.no_grad()
    def ecapa_embed(w, wl=None):
        return F.normalize(ecapa.encode_batch(w, wl).squeeze(1).float(), dim=-1)

    with torch.no_grad():
        EMB_DIM = ecapa_embed(torch.zeros(1, SR, device=DEVICE), torch.ones(1, device=DEVICE)).shape[-1]
    print(f"ECAPA dim: {EMB_DIM}")

    # ---- Wave-U-Net (frozen, loaded from v6 checkpoint) ----
    MODEL_ID = "wrice/waveunet-vctk-48khz-audiomentations"
    print("Loading Wave-U-Net ...")
    with open(hf_hub_download(MODEL_ID, "config.json")) as f: cd = json.load(f)
    cfg = WaveUNetConfig(**cd); MODEL_SR = cd["sample_rate"]; MAX_LEN = cd["max_length"]
    enhancer = WaveUNetModel(cfg).to(DEVICE)

    if not os.path.isfile(ENH_CKPT):
        raise FileNotFoundError(f"Enhancer checkpoint not found: {ENH_CKPT}")
    ck = torch.load(ENH_CKPT, map_location="cpu", weights_only=False)
    enh_state = ck.get("enhancer_state_dict", ck.get("state_dict"))
    if enh_state is None:
        raise RuntimeError("No enhancer state dict in checkpoint")

    # Remap keys
    new_sd = {}
    for k, v in enh_state.items():
        k2 = k
        if (".encoder_layers." in k or ".decoder_layers." in k) and ".batch_norm." in k:
            k2 = k2.replace(".batch_norm.", ".norm.norm.")
        m = re.match(r"^(.*\.middle\.\d+)\.(weight|bias|running_mean|running_var|num_batches_tracked)$", k2)
        if m: k2 = f"{m.group(1)}.norm.{m.group(2)}"
        new_sd[k2] = v
    missing, unexpected = enhancer.load_state_dict(new_sd, strict=False)
    print(f"  WUN loaded: missing={len(missing)} unexpected={len(unexpected)}")
    print(f"  Enhancer best_val: {ck.get('best_val', 'N/A')}")
    enhancer.eval()
    for p in enhancer.parameters(): p.requires_grad = False

    # ---- Chunked enhancement (frozen) ----
    @torch.no_grad()
    def enhance_chunked(wav, wl, cbs=None):
        if cbs is None: cbs = CHUNK_BS
        B, Tm = wav.shape
        r16 = (wl.detach().cpu().float() * Tm).round().long()
        r48 = (r16.float() * MODEL_SR / SR).round().long()
        wav48 = torchaudio.functional.resample(wav, SR, MODEL_SR)
        T48m = wav48.shape[-1]

        chunks = []; owners = []
        for b in range(B):
            Lb = int(r48[b].item())
            if Lb <= 0: continue
            w = wav48[b, :Lb]
            pad = (MAX_LEN - w.numel() % MAX_LEN) % MAX_LEN
            wp = F.pad(w, (0, pad)) if pad else w
            n = wp.numel() // MAX_LEN
            for i in range(n):
                chunks.append(wp[i * MAX_LEN:(i + 1) * MAX_LEN]); owners.append((b, i))

        if not chunks: return wav

        chunks = torch.stack(chunks).unsqueeze(1)
        enh_chunks = []
        with make_autocast(USE_CUDA):
            for i in range(0, chunks.shape[0], cbs):
                c = chunks[i:i + cbs]
                out = enhancer(c)
                noise = _unwrap(out).reshape(c.shape[0], -1)[:, :MAX_LEN].float()
                enh_chunks.append(c.squeeze(1).float() - noise)
        enh_chunks = torch.cat(enh_chunks, dim=0)

        per = [[] for _ in range(B)]
        for idx, (b, _) in enumerate(owners): per[b].append(enh_chunks[idx])

        out_list = []
        for b in range(B):
            e = torch.cat(per[b], dim=0)[:int(r48[b].item())] if per[b] else torch.zeros(0, device=wav.device)
            if e.numel() < T48m: e = F.pad(e, (0, T48m - e.numel()))
            else: e = e[:T48m]
            out_list.append(e)
        e48 = torch.stack(out_list, dim=0)
        e16 = torchaudio.functional.resample(e48, MODEL_SR, SR)
        if e16.shape[-1] > Tm: e16 = e16[..., :Tm]
        elif e16.shape[-1] < Tm: e16 = F.pad(e16, (0, Tm - e16.shape[-1]))
        return e16

    # =========================================================================
    # THREE FUSION ARCHITECTURES
    # =========================================================================
    # All three share the same I/O contract: (en, ee) -> (fused, beta).
    # All three use:
    #   beta  = sigmoid(gate_head(features))    -> learned blend weight
    #   delta = proj_out(features)              -> zero-init residual correction
    #   fused = normalize((1-beta)*en + beta*ee + delta)
    # The differences are in HOW the features that drive beta and delta
    # are computed:
    #   MLP:   concat + 2-layer MLP
    #   SA:    two-token self-attention over [en, ee]
    #   CA:    cross-attention Q=raw, K,V=enhanced
    # =========================================================================

    class MLPFusion(nn.Module):
        """Simple MLP on concatenated embeddings."""
        def __init__(self, dim=EMB_DIM, hidden=384, dropout=0.1):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(dim * 2, hidden),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden, hidden),
                nn.GELU(),
                nn.Dropout(dropout),
            )
            self.gate_head = nn.Linear(hidden, 1)
            nn.init.zeros_(self.gate_head.weight); nn.init.zeros_(self.gate_head.bias)
            self.proj_out = nn.Linear(hidden, dim)
            nn.init.zeros_(self.proj_out.weight); nn.init.zeros_(self.proj_out.bias)

        def forward(self, en, ee, return_beta=False):
            x = torch.cat([en, ee], dim=-1)
            h = self.net(x)
            beta = torch.sigmoid(self.gate_head(h))
            delta = self.proj_out(h)
            blended = (1.0 - beta) * en + beta * ee
            out = F.normalize(blended + delta, dim=-1)
            if return_beta: return out, beta.squeeze(-1)
            return out

    class SelfAttentionFusion(nn.Module):
        """Two-token self-attention over [en, ee]."""
        def __init__(self, dim=EMB_DIM, nh=SA_NUM_HEADS, nl=SA_NUM_LAYERS, dp=SA_DROPOUT):
            super().__init__()
            self.proj = nn.Linear(dim, dim)
            self.pos_emb = nn.Parameter(torch.zeros(1, 2, dim))
            layer = nn.TransformerEncoderLayer(d_model=dim, nhead=nh, dim_feedforward=dim * 4,
                                                dropout=dp, batch_first=True, activation="gelu", norm_first=True)
            self.encoder = nn.TransformerEncoder(layer, num_layers=nl)
            self.norm = nn.LayerNorm(dim)
            self.gate_head = nn.Linear(dim, 1)
            nn.init.zeros_(self.gate_head.weight); nn.init.zeros_(self.gate_head.bias)
            self.proj_out = nn.Linear(dim, dim)
            nn.init.zeros_(self.proj_out.weight); nn.init.zeros_(self.proj_out.bias)

        def forward(self, en, ee, return_beta=False):
            n = self.proj(en); e = self.proj(ee)
            x = torch.stack([n, e], dim=1).contiguous() + self.pos_emb
            h = self.encoder(x)
            pooled = self.norm(h.mean(dim=1))
            beta = torch.sigmoid(self.gate_head(pooled))
            delta = self.proj_out(pooled)
            blended = (1.0 - beta) * en + beta * ee
            out = F.normalize(blended + delta, dim=-1)
            if return_beta: return out, beta.squeeze(-1)
            return out

    class CrossAttentionFusion(nn.Module):
        """Q from raw, K/V from enhanced. Query attends over enhanced."""
        def __init__(self, dim=EMB_DIM, nh=CA_NUM_HEADS, dp=CA_DROPOUT):
            super().__init__()
            self.q_proj = nn.Linear(dim, dim)
            self.k_proj = nn.Linear(dim, dim)
            self.v_proj = nn.Linear(dim, dim)
            self.out_proj = nn.Linear(dim, dim)
            self.norm1 = nn.LayerNorm(dim)
            self.norm2 = nn.LayerNorm(dim)
            self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=nh, dropout=dp, batch_first=True)
            self.mlp = nn.Sequential(
                nn.Linear(dim, dim * 2), nn.GELU(), nn.Dropout(dp),
                nn.Linear(dim * 2, dim), nn.Dropout(dp),
            )
            # combine raw + cross-attn output into features
            self.fuse_proj = nn.Linear(dim * 2, dim)
            self.gate_head = nn.Linear(dim, 1)
            nn.init.zeros_(self.gate_head.weight); nn.init.zeros_(self.gate_head.bias)
            self.proj_out = nn.Linear(dim, dim)
            nn.init.zeros_(self.proj_out.weight); nn.init.zeros_(self.proj_out.bias)

        def forward(self, en, ee, return_beta=False):
            # en: (B, D) raw, ee: (B, D) enhanced
            q = self.q_proj(en).unsqueeze(1)   # (B, 1, D)
            k = self.k_proj(ee).unsqueeze(1)   # (B, 1, D)
            v = self.v_proj(ee).unsqueeze(1)   # (B, 1, D)
            attn_out, _ = self.attn(q, k, v, need_weights=False)
            attn_out = self.out_proj(attn_out.squeeze(1))   # (B, D)
            x = self.norm1(en + attn_out)
            x = self.norm2(x + self.mlp(x))
            # combine raw and cross-attended features
            pooled = self.fuse_proj(torch.cat([en, x], dim=-1))
            beta = torch.sigmoid(self.gate_head(pooled))
            delta = self.proj_out(pooled)
            blended = (1.0 - beta) * en + beta * ee
            out = F.normalize(blended + delta, dim=-1)
            if return_beta: return out, beta.squeeze(-1)
            return out

    # =========================================================================
    # DATASET + SAMPLER
    # =========================================================================
    class VoxDataset(Dataset):
        def __init__(self, files, spk_to_idx, noise_banks, snr_range):
            self.files = files
            self.labels = [spk_to_idx[extract_spk(f)] for f in files]
            self.noise_banks = noise_banks
            self.noise_types = sorted(noise_banks.keys())
            self.snr_range = snr_range
        def __len__(self): return len(self.files)
        def _pick_noise_file(self):
            nt = random.choice(self.noise_types)
            return random.choice(self.noise_banks[nt])
        def __getitem__(self, idx):
            seg = int(random.uniform(MIN_SEC, MAX_SEC) * SR)
            cf = load_wav(self.files[idx], SR); n = cf.numel()
            clean = cf[random.randint(0, n - seg):][:seg].clone() if n >= seg else F.pad(cf, (0, seg - n))
            if random.random() < CLEAN_FRAC:
                return clean.clone(), clean.clone(), self.labels[idx], CLEAN_SNR
            nf = load_wav(self._pick_noise_file(), SR); n = nf.numel()
            noise = nf[random.randint(0, n - seg):][:seg].clone() if n >= seg else F.pad(nf, (0, seg - n))
            snr = random.uniform(*self.snr_range)
            noisy, clean, _ = mix_at_snr(clean, noise, snr)
            return noisy, clean, self.labels[idx], snr

    def collate_simple(batch):
        nl, cl, lb, sn = zip(*batch)
        L = torch.tensor([w.numel() for w in nl], dtype=torch.long)
        m = int(L.max().item())
        return (torch.stack([F.pad(w, (0, m - w.numel())) for w in nl]),
                torch.stack([F.pad(w, (0, m - w.numel())) for w in cl]),
                torch.tensor(lb, dtype=torch.long), L.float() / m,
                torch.tensor(sn, dtype=torch.float32))

    class PKBatchSampler(Sampler):
        def __init__(self, ds, p, k, steps):
            self.p = p; self.k = k; self.steps = steps
            by = defaultdict(list)
            for i, lab in enumerate(ds.labels): by[lab].append(i)
            self.by = {lab: i for lab, i in by.items() if len(i) >= k}
            self.spks = sorted(self.by.keys())
        def __iter__(self):
            for _ in range(self.steps):
                spks = random.sample(self.spks, self.p); batch = []
                for s in spks: batch.extend(random.sample(self.by[s], self.k))
                yield batch
        def __len__(self): return self.steps

    # ---- Speaker split ----
    print("\nScanning VoxCeleb1 dev ...")
    by_spk = defaultdict(list)
    for f in list_audio(DEV_ROOT): by_spk[extract_spk(f)].append(f)
    all_spks = sorted([s for s, v in by_spk.items() if len(v) >= K_UTTS])
    random.seed(42); random.shuffle(all_spks)
    val_spks = set(all_spks[:N_VAL_SPEAKERS]); train_spks = set(all_spks[N_VAL_SPEAKERS:])
    train_files = [f for s in train_spks for f in by_spk[s]]
    spk_to_idx = {s: i for i, s in enumerate(sorted(train_spks))}
    print(f"Train: {len(train_spks)} spk | Val: {len(val_spks)} spk | Utter: {len(train_files)}")

    def build_val_trials():
        trials = []; rng = random.Random(123); vl = sorted(val_spks)
        for spk in vl:
            utts = sorted(by_spk[spk])
            if len(utts) < 2: continue
            enr = utts[0]
            for tst in utts[1:min(1 + N_VAL_UTTS_PER_SPK, len(utts))]: trials.append((enr, tst, 1))
            others = [s for s in vl if s != spk]
            for _ in range(min(N_VAL_UTTS_PER_SPK, len(others))):
                imp = rng.choice(others); trials.append((enr, rng.choice(by_spk[imp]), 0))
        return trials
    val_trials = build_val_trials()
    VAL_ENR = sorted({e for e, _, _ in val_trials}); VAL_TST = sorted({t for _, t, _ in val_trials})
    print(f"Val trials: {len(val_trials)}")

    # =========================================================================
    # LOSSES
    # =========================================================================
    def triplet_loss(anchor, fused, labels, margin=TRIPLET_MARGIN):
        B = fused.size(0); dev = fused.device
        d_pos = 1.0 - (anchor * fused).sum(-1)
        sim = anchor @ fused.t(); d_mat = 1.0 - sim
        same = labels.unsqueeze(0) == labels.unsqueeze(1); diff = ~same
        d_neg = d_mat.masked_fill(~diff, float("inf"))
        valid = (d_neg > d_pos.unsqueeze(1)) & (d_neg < (d_pos.unsqueeze(1) + margin))
        d_semi = d_neg.masked_fill(~valid, float("inf"))
        has_semi = valid.any(dim=1)
        bs, _ = d_semi.min(dim=1); hr, _ = d_neg.min(dim=1)
        d_sel = torch.where(has_semi, bs, hr)
        losses = F.relu(d_pos - d_sel + margin)
        va = torch.isfinite(d_sel)
        if va.sum() == 0: return torch.tensor(0.0, device=dev, requires_grad=True), 0
        return losses[va].mean(), int((losses[va] > 0).sum().item())

    def fus_preserve(fused, en, snrs, th=FUS_PRESERVE_SNR_THRESHOLD, tp=FUS_PRESERVE_SNR_TEMP):
        w = torch.sigmoid((snrs - th) / tp)
        per = 1.0 - (fused * en).sum(-1)
        return (per * w).sum() / w.sum().clamp_min(1e-6)

    # =========================================================================
    # VALIDATION CACHE — computed ONCE, reused across all epochs and fusions
    # =========================================================================
    # The val set is fixed (fixed speakers, fixed trial list, deterministic
    # noise per (nt, snr, path)). So the enrollment embeddings, noisy test
    # embeddings, and enhanced test embeddings never change across calls.
    # Precompute them once; each subsequent validation only runs the fusion
    # forward pass (sub-second).
    # =========================================================================
    print("\n" + "=" * 78)
    print("PRECOMPUTING VALIDATION CACHE (one-time cost)")
    print("=" * 78)

    # ---- 1. Enrollment embeddings (clean, no noise) ----
    print("  [1/3] Caching val enrollment embeddings ...")
    VAL_ENR_EMB = {}
    for p in tqdm(VAL_ENR, desc="val_enr"):
        w = load_wav(p, SR).unsqueeze(0).to(DEVICE)
        wl = torch.ones(1, device=DEVICE)
        VAL_ENR_EMB[p] = ecapa_embed(w, wl).cpu()[0]
    if USE_CUDA: torch.cuda.empty_cache()
    print(f"        {len(VAL_ENR_EMB)} enrollment embeddings cached")

    # ---- 2 & 3. Per-(nt, snr) noisy and enhanced test embeddings ----
    print(f"  [2/3] Caching noisy + enhanced test embeddings across "
          f"{len(FUS_VAL_SNRS) * 3} conditions ...")

    VAL_TEST_RAW = {}   # (nt, snr) -> {path: embedding}  raw noisy probe
    VAL_TEST_ENH = {}   # (nt, snr) -> {path: embedding}  enhanced probe
    VAL_RAW_EER = {}    # (nt, snr) -> EER using raw probe
    VAL_ENH_EER = {}    # (nt, snr) -> EER using enhanced probe

    conditions_for_cache = [(nt, snr) for nt in ["babble", "music", "noise"]
                            for snr in FUS_VAL_SNRS]

    for nt, snr in tqdm(conditions_for_cache, desc="val_cache"):
        nf = noise_train_banks[nt]
        tst_raw = {}; tst_enh = {}
        for i in range(0, len(VAL_TST), EVAL_BATCH):
            b = VAL_TST[i:i + EVAL_BATCH]
            wavs = [load_wav(p, SR) for p in b]
            mixed = []
            for p, w in zip(b, wavs):
                r = random.Random(sseed("fusval", nt, snr, p))
                npx = r.choice(nf); nz = load_wav(npx, SR)
                no, _, _ = mix_det(w, nz, snr, ("fusval", nt, snr, p))
                mixed.append(no)
            L = torch.tensor([w.numel() for w in mixed], dtype=torch.long)
            ml = int(L.max().item()); wl = (L.float() / ml).to(DEVICE)
            pad = torch.stack([F.pad(w, (0, ml - w.numel())) for w in mixed]).to(DEVICE)

            en = ecapa_embed(pad, wl).cpu()
            enh_wav = enhance_chunked(pad, wl).float()
            ee = ecapa_embed(enh_wav, wl).cpu()

            for j, p in enumerate(b):
                tst_raw[p] = en[j]
                tst_enh[p] = ee[j]
            del pad, enh_wav
        VAL_TEST_RAW[(nt, snr)] = tst_raw
        VAL_TEST_ENH[(nt, snr)] = tst_enh

        # EER for both baselines on this condition
        sc = []; lb = []
        for e, t, l in val_trials:
            ee_ = VAL_ENR_EMB[e]
            te_ = tst_raw[t]
            sc.append(F.cosine_similarity(ee_.unsqueeze(0), te_.unsqueeze(0)).item())
            lb.append(l)
        VAL_RAW_EER[(nt, snr)] = compute_eer(sc, lb)

        sc = []; lb = []
        for e, t, l in val_trials:
            ee_ = VAL_ENR_EMB[e]
            te_ = tst_enh[t]
            sc.append(F.cosine_similarity(ee_.unsqueeze(0), te_.unsqueeze(0)).item())
            lb.append(l)
        VAL_ENH_EER[(nt, snr)] = compute_eer(sc, lb)

        if USE_CUDA: torch.cuda.empty_cache()

    print(f"  [3/3] Cache complete")
    print(f"        Raw baseline mean EER: {np.mean(list(VAL_RAW_EER.values())):.2f}%")
    print(f"        Enh baseline mean EER: {np.mean(list(VAL_ENH_EER.values())):.2f}%")

    # Pre-stack tensors per condition for fast batched fusion forward
    VAL_PATHS_ORDERED = list(VAL_TST)   # fixed order
    VAL_RAW_STACKED = {}   # (nt, snr) -> tensor [N, D]
    VAL_ENH_STACKED = {}   # (nt, snr) -> tensor [N, D]
    for key in VAL_TEST_RAW:
        VAL_RAW_STACKED[key] = torch.stack([VAL_TEST_RAW[key][p] for p in VAL_PATHS_ORDERED])
        VAL_ENH_STACKED[key] = torch.stack([VAL_TEST_ENH[key][p] for p in VAL_PATHS_ORDERED])

    # ---- Fast validation using the cache ----
    @torch.inference_mode()
    def validate_fusion(fusion_model):
        """Fast validation: only the fusion forward pass per condition."""
        fusion_model.eval()
        per = {}
        beta_sum = {}; beta_cnt = {}

        for nt in ["babble", "music", "noise"]:
            for snr in FUS_VAL_SNRS:
                en = VAL_RAW_STACKED[(nt, snr)].to(DEVICE)
                ee = VAL_ENH_STACKED[(nt, snr)].to(DEVICE)
                f, beta = fusion_model(en, ee, return_beta=True)
                f = f.cpu(); beta = beta.cpu()

                tst_fused = {p: f[i] for i, p in enumerate(VAL_PATHS_ORDERED)}

                sc = []; lb = []
                for e, t, l in val_trials:
                    ee_ = VAL_ENR_EMB[e]
                    te_ = tst_fused[t]
                    sc.append(F.cosine_similarity(ee_.unsqueeze(0), te_.unsqueeze(0)).item())
                    lb.append(l)

                fe = compute_eer(sc, lb)
                re = VAL_RAW_EER[(nt, snr)]
                bm = beta.mean().item()
                per[(nt, snr)] = (fe, re, bm)

        fusion_model.train()
        ratios = [v[0] / max(v[1], 0.5) for v in per.values()]
        return float(np.mean(ratios)), per

    # =========================================================================
    # TRAIN ONE FUSION
    # =========================================================================
    def train_fusion(fusion_model, name):
        print("\n" + "=" * 78)
        print(f"TRAINING FUSION: {name}")
        print("=" * 78)
        n_params = sum(p.numel() for p in fusion_model.parameters())
        print(f"  Params: {n_params:,}")

        train_ds = VoxDataset(train_files, spk_to_idx, noise_train_banks, SNR_RANGE)
        sampler = PKBatchSampler(train_ds, P_SPEAKERS, K_UTTS, FUS_STEPS_PER_EP)
        nw = 4 if _MP_CTX is not None else 0
        lk = dict(batch_sampler=sampler, num_workers=nw, pin_memory=True,
                  persistent_workers=(nw > 0), collate_fn=collate_simple)
        if _MP_CTX is not None: lk["multiprocessing_context"] = _MP_CTX
        loader = DataLoader(train_ds, **lk)

        opt = torch.optim.AdamW(fusion_model.parameters(), lr=FUS_LR, weight_decay=1e-5)
        total_steps = FUS_EPOCHS * FUS_STEPS_PER_EP
        def lr_lam(s):
            if s < FUS_WARMUP_STEPS: return s / max(1, FUS_WARMUP_STEPS)
            p = (s - FUS_WARMUP_STEPS) / max(1, total_steps - FUS_WARMUP_STEPS)
            return 0.5 * (1.0 + math.cos(math.pi * p))
        sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lam)
        scaler = make_scaler(USE_CUDA)

        best_path = os.path.join(FUS_CKPT_DIR, f"{name}_best.pt")
        best_rel = float("inf")
        best_state = None

        t0 = time.time()
        for epoch in range(FUS_EPOCHS):
            fusion_model.train()
            m = defaultdict(float)
            pbar = tqdm(loader, desc=f"{name} {epoch + 1}/{FUS_EPOCHS}")
            opt.zero_grad(set_to_none=True)

            for step_i, (noisy, clean, labels, wl, snrs) in enumerate(pbar):
                noisy = noisy.to(DEVICE, non_blocking=True)
                clean = clean.to(DEVICE, non_blocking=True)
                labels = labels.to(DEVICE, non_blocking=True)
                wl = wl.to(DEVICE, non_blocking=True)
                snrs = snrs.to(DEVICE, non_blocking=True)

                with torch.no_grad():
                    ec = ecapa_embed(clean, wl)
                    en = ecapa_embed(noisy, wl)
                    ge = enhance_chunked(noisy, wl).float()
                    ee = ecapa_embed(ge, wl)

                with make_autocast(USE_CUDA):
                    f, beta = fusion_model(en.float(), ee.float(), return_beta=True)

                lt, na = triplet_loss(ec, f, labels)
                lp = fus_preserve(f, en, snrs)
                total = lt + LAMBDA_FUS_PRESERVE * lp
                if not torch.isfinite(total): continue

                scaler.scale(total).backward(); scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(fusion_model.parameters(), FUS_GRAD_CLIP)
                prev = scaler.get_scale()
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
                if scaler.get_scale() >= prev: sched.step()

                m["total"] += total.item(); m["triplet"] += lt.item()
                m["preserve"] += lp.item(); m["active"] += na
                m["beta_mean"] += beta.mean().item()

                pbar.set_postfix({"L": f"{total.item():.3f}", "trip": f"{lt.item():.3f}",
                                  "prsv": f"{lp.item():.3f}", "act": f"{na}",
                                  "β": f"{beta.mean().item():.2f}"})

            n = max(1, len(pbar))
            print(f"\n{name} Epoch {epoch + 1}: " + " | ".join(f"{k}={v / n:.4f}" for k, v in m.items()))

            do_eval = ((epoch + 1) % 2 == 0) or (epoch == FUS_EPOCHS - 1)
            if do_eval:
                rv, pc = validate_fusion(fusion_model)
                print(f"  Mean ratio: {rv:.3f}")
                for nt in ["babble", "music", "noise"]:
                    cells = []
                    for snr in FUS_VAL_SNRS:
                        if (nt, snr) in pc:
                            fe, re, bm = pc[(nt, snr)]
                            cells.append(f"{snr:+4.0f}dB={fe:5.1f}/{re:5.1f}|β={bm:.2f}")
                    print(f"    {nt:8s}  " + "  ".join(cells))

                if rv < best_rel:
                    best_rel = rv
                    best_state = {k: v.detach().cpu().clone() for k, v in fusion_model.state_dict().items()}
                    torch.save({
                        "model_state_dict": fusion_model.state_dict(),
                        "best_rel": best_rel,
                        "arch": name,
                        "params": n_params,
                    }, best_path)
                    print(f"  ✓ New best ({rv:.3f})")
                    if _DRIVE_ENABLED:
                        try: fs.put(best_path, f"{_DRIVE_REMOTE}/{name}_best.pt")
                        except Exception as e: print(f"  [drive] upload failed: {e}")

                if USE_CUDA: torch.cuda.empty_cache()
                gc.collect()

        print(f"\n{name} training time: {(time.time() - t0) / 60:.1f} min")

        if best_state is not None:
            fusion_model.load_state_dict(best_state)
        elif os.path.isfile(best_path):
            ck = torch.load(best_path, map_location="cpu", weights_only=False)
            fusion_model.load_state_dict(ck["model_state_dict"])
            best_rel = ck.get("best_rel", best_rel)

        fusion_model.eval()
        print(f"{name} best val ratio: {best_rel:.3f}")
        return fusion_model, best_rel

    # =========================================================================
    # TRAIN ALL THREE
    # =========================================================================
    mlp = MLPFusion(dim=EMB_DIM).to(DEVICE)
    sa = SelfAttentionFusion(dim=EMB_DIM).to(DEVICE)
    ca = CrossAttentionFusion(dim=EMB_DIM).to(DEVICE)

    mlp, mlp_rel = train_fusion(mlp, "mlp")
    sa, sa_rel = train_fusion(sa, "sa")
    ca, ca_rel = train_fusion(ca, "ca")

    print("\n" + "=" * 78)
    print("FUSION TRAINING SUMMARY")
    print("=" * 78)
    print(f"  MLP          : val ratio = {mlp_rel:.3f}")
    print(f"  Self-Attn    : val ratio = {sa_rel:.3f}")
    print(f"  Cross-Attn   : val ratio = {ca_rel:.3f}")

    # =========================================================================
    # FULL TEST EVALUATION
    # =========================================================================
    print("\n" + "=" * 78 + "\nFULL TEST SWEEP\n" + "=" * 78)
    if not os.path.isfile(TRIAL_FILE):
        urllib.request.urlretrieve(
            "https://www.robots.ox.ac.uk/~vgg/data/voxceleb/meta/veri_test2.txt", TRIAL_FILE)

    def load_trials(path):
        out = []
        with open(path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 3: out.append((parts[1], parts[2], int(parts[0])))
        return out
    def resolve(rel):
        rel = rel.replace("\\", "/").lstrip("/")
        for base in (TEST_ROOT, os.path.join(TEST_ROOT, "wav"), "/marimo/voxceleb1_test"):
            p = os.path.join(base, rel)
            if os.path.isfile(p): return p
        mm = list(Path(TEST_ROOT).rglob(os.path.basename(rel)))
        return str(mm[0]) if mm else None

    all_trials = load_trials(TRIAL_FILE)
    resolved = []
    for e, t, lab in all_trials:
        ep, tp = resolve(e), resolve(t)
        if ep and tp: resolved.append((ep, tp, lab))
    assert not [(e, t) for e, t, _ in resolved if e == t]
    ue = sorted({e for e, _, _ in resolved}); ut = sorted({t for _, t, _ in resolved})
    print(f"Resolved: {len(resolved)}  Enroll: {len(ue)}  Test: {len(ut)}")

    @torch.inference_mode()
    def embed_clean(paths, batch_size=16):
        cache = {}
        for i in range(0, len(paths), batch_size):
            batch = paths[i:i + batch_size]
            wavs = [load_wav(p, SR) for p in batch]
            L = torch.tensor([w.numel() for w in wavs], dtype=torch.long)
            ml = int(L.max().item()); wl = (L.float() / ml).to(DEVICE)
            pad = torch.stack([F.pad(w, (0, ml - w.numel())) for w in wavs]).to(DEVICE)
            emb = ecapa_embed(pad, wl).cpu()
            for j, p in enumerate(batch): cache[p] = emb[j]
        return cache

    @torch.inference_mode()
    def embed_condition(paths, nt=None, snr=None, noise_files=None, batch_size=16, tag="test"):
        """Compute raw, enhanced, and all 3 fused embeddings in one pass."""
        raw_c, enh_c, mlp_c, sa_c, ca_c = {}, {}, {}, {}, {}
        for i in range(0, len(paths), batch_size):
            batch = paths[i:i + batch_size]
            wavs = [load_wav(p, SR) for p in batch]
            if snr is not None:
                mixed = []
                for p, w in zip(batch, wavs):
                    r = random.Random(sseed(tag, nt, snr, p))
                    npx = r.choice(noise_files); nz = load_wav(npx, SR)
                    no, _, _ = mix_det(w, nz, snr, (tag, nt, snr, p))
                    mixed.append(no)
                wavs = mixed
            L = torch.tensor([w.numel() for w in wavs], dtype=torch.long)
            ml = int(L.max().item()); wl = (L.float() / ml).to(DEVICE)
            pad = torch.stack([F.pad(w, (0, ml - w.numel())) for w in wavs]).to(DEVICE)

            en = ecapa_embed(pad, wl)
            enh_wav = enhance_chunked(pad, wl).float()
            ee = ecapa_embed(enh_wav, wl)

            f_mlp = mlp(en, ee)
            f_sa = sa(en, ee)
            f_ca = ca(en, ee)

            en_c, ee_c = en.cpu(), ee.cpu()
            f_mlp_c = f_mlp.cpu(); f_sa_c = f_sa.cpu(); f_ca_c = f_ca.cpu()
            for j, p in enumerate(batch):
                raw_c[p] = en_c[j]; enh_c[p] = ee_c[j]
                mlp_c[p] = f_mlp_c[j]; sa_c[p] = f_sa_c[j]; ca_c[p] = f_ca_c[j]
            del pad, enh_wav
        if USE_CUDA: torch.cuda.empty_cache()
        return raw_c, enh_c, mlp_c, sa_c, ca_c

    print("\nCaching clean enrollment embeddings ...")
    enr_cache = embed_clean(ue, batch_size=16)
    print(f"  {len(enr_cache)} cached")

    conditions = [("clean", None, None)]
    for nt, _ in NOISE_SUBDIRS:
        for snr in EVAL_SNRS: conditions.append((nt, nt, snr))
    print(f"\nEvaluating {len(conditions)} conditions ...")
    results = []
    t0 = time.time()
    for cond, nt, snr in tqdm(conditions, desc="conditions"):
        nf = noise_test_banks[nt] if snr is not None else None
        raw_c, enh_c, mlp_c, sa_c, ca_c = embed_condition(ut, nt=nt, snr=snr, noise_files=nf, batch_size=16)

        s_raw, s_enh, s_mlp, s_sa, s_ca, lb = [], [], [], [], [], []
        for e, t, l in resolved:
            ee = enr_cache[e].to(DEVICE)
            s_raw.append(F.cosine_similarity(ee.unsqueeze(0), raw_c[t].unsqueeze(0).to(DEVICE)).item())
            s_enh.append(F.cosine_similarity(ee.unsqueeze(0), enh_c[t].unsqueeze(0).to(DEVICE)).item())
            s_mlp.append(F.cosine_similarity(ee.unsqueeze(0), mlp_c[t].unsqueeze(0).to(DEVICE)).item())
            s_sa.append(F.cosine_similarity(ee.unsqueeze(0), sa_c[t].unsqueeze(0).to(DEVICE)).item())
            s_ca.append(F.cosine_similarity(ee.unsqueeze(0), ca_c[t].unsqueeze(0).to(DEVICE)).item())
            lb.append(l)
        s_raw = np.array(s_raw); s_enh = np.array(s_enh)
        s_mlp = np.array(s_mlp); s_sa = np.array(s_sa); s_ca = np.array(s_ca)
        lb = np.array(lb)

        results.append({
            "condition": cond,
            "noise_type": nt if snr is not None else "clean",
            "snr": snr,
            "raw_eer": compute_eer(s_raw, lb),
            "enh_eer": compute_eer(s_enh, lb),
            "mlp_eer": compute_eer(s_mlp, lb),
            "sa_eer": compute_eer(s_sa, lb),
            "ca_eer": compute_eer(s_ca, lb),
        })
        gc.collect()
        if USE_CUDA: torch.cuda.empty_cache()

    print(f"\nSweep time: {(time.time() - t0) / 60:.1f} min")
    df = pd.DataFrame(results)
    df["delta_enh"] = df["enh_eer"] - df["raw_eer"]
    df["delta_mlp"] = df["mlp_eer"] - df["raw_eer"]
    df["delta_sa"] = df["sa_eer"] - df["raw_eer"]
    df["delta_ca"] = df["ca_eer"] - df["raw_eer"]
    df.to_csv("/marimo/fusion_comparison_results.csv", index=False)
    if _DRIVE_ENABLED:
        try: fs.put("/marimo/fusion_comparison_results.csv", f"{_DRIVE_REMOTE}/fusion_comparison_results.csv")
        except Exception as e: print(f"[drive] upload failed: {e}")

    # =========================================================================
    # RESULTS
    # =========================================================================
    def piv(c):
        p = df.pivot_table(index="noise_type", columns="snr", values=c, dropna=False)
        return p.reindex(["clean"] + [c for c in p.index if c != "clean"])

    print("\n" + "=" * 80)
    print("EER (%) — Raw ECAPA")
    print("=" * 80); print(piv("raw_eer").round(2))
    print("\n" + "=" * 80)
    print("EER (%) — Enhancer-only")
    print("=" * 80); print(piv("enh_eer").round(2))
    print("\n" + "=" * 80)
    print("EER (%) — MLP Fusion")
    print("=" * 80); print(piv("mlp_eer").round(2))
    print("\n" + "=" * 80)
    print("EER (%) — Self-Attention Fusion")
    print("=" * 80); print(piv("sa_eer").round(2))
    print("\n" + "=" * 80)
    print("EER (%) — Cross-Attention Fusion")
    print("=" * 80); print(piv("ca_eer").round(2))

    # Summary
    print("\n" + "=" * 80)
    print("SUMMARY BY NOISE TYPE (avg across SNRs)")
    print("=" * 80)
    nd = df[df["condition"] != "clean"]
    print(f"\n{'Noise type':<12} {'Raw':>8} {'Enh':>8} {'MLP':>8} {'SA':>8} {'CA':>8}")
    print("-" * 60)
    for nt in ["babble", "music", "noise"]:
        s = nd[nd["noise_type"] == nt]
        print(f"{nt:<12} {s['raw_eer'].mean():>8.2f} {s['enh_eer'].mean():>8.2f} "
              f"{s['mlp_eer'].mean():>8.2f} {s['sa_eer'].mean():>8.2f} {s['ca_eer'].mean():>8.2f}")
    print("-" * 60)
    print(f"{'ALL':<12} {nd['raw_eer'].mean():>8.2f} {nd['enh_eer'].mean():>8.2f} "
          f"{nd['mlp_eer'].mean():>8.2f} {nd['sa_eer'].mean():>8.2f} {nd['ca_eer'].mean():>8.2f}")

    pp = nd[nd["snr"].isin(PAPER_SNRS)]
    print(f"\nPaper subset (0 to -20 dB):")
    print(f"{'':<12} {pp['raw_eer'].mean():>8.2f} {pp['enh_eer'].mean():>8.2f} "
          f"{pp['mlp_eer'].mean():>8.2f} {pp['sa_eer'].mean():>8.2f} {pp['ca_eer'].mean():>8.2f}")

    cl = df[df["condition"] == "clean"].iloc[0]
    print(f"\nClean:")
    print(f"{'':<12} {cl['raw_eer']:>8.2f} {cl['enh_eer']:>8.2f} "
          f"{cl['mlp_eer']:>8.2f} {cl['sa_eer']:>8.2f} {cl['ca_eer']:>8.2f}")

    # Winner identification
    print("\n" + "=" * 80)
    print("BEST MODEL PER REGION")
    print("=" * 80)
    for label, region_df in [("All noisy", nd), ("Paper subset", pp)]:
        scores = {
            "Raw": region_df["raw_eer"].mean(),
            "Enh": region_df["enh_eer"].mean(),
            "MLP": region_df["mlp_eer"].mean(),
            "SA":  region_df["sa_eer"].mean(),
            "CA":  region_df["ca_eer"].mean(),
        }
        best = min(scores, key=scores.get)
        print(f"  {label:15s}: best = {best} ({scores[best]:.2f}%)")

    # Delta summary
    print("\n" + "=" * 80)
    print("IMPROVEMENT OVER RAW (Δ EER, negative = better)")
    print("=" * 80)
    print(f"{'':<12} {'Enh':>8} {'MLP':>8} {'SA':>8} {'CA':>8}")
    print("-" * 44)
    for nt in ["babble", "music", "noise"]:
        s = nd[nd["noise_type"] == nt]
        print(f"{nt:<12} {s['delta_enh'].mean():>+8.2f} {s['delta_mlp'].mean():>+8.2f} "
              f"{s['delta_sa'].mean():>+8.2f} {s['delta_ca'].mean():>+8.2f}")
    print("-" * 44)
    print(f"{'ALL':<12} {nd['delta_enh'].mean():>+8.2f} {nd['delta_mlp'].mean():>+8.2f} "
          f"{nd['delta_sa'].mean():>+8.2f} {nd['delta_ca'].mean():>+8.2f}")

    return df, {"mlp": mlp, "sa": sa, "ca": ca}


df_comparison, fusion_models = train_and_evaluate_fusions()

# Tables from the v6-fusion-comparison-v1 run

## MUSAN split

| Noise type | Train | Test |
|---|---:|---:|
| babble | 299 | 127 |
| music | 462 | 198 |
| noise | 651 | 279 |

## Validation cache baselines

| Baseline | Mean EER |
|---|---:|
| Raw ECAPA | 10.59% |
| Enhancer-only | 8.95% |

## Fusion training summary (best validation ratio)

| Architecture | Params | Best val ratio |
|---|---:|---:|
| MLP | 369,985 | 0.541 |
| Self-Attention | 964,801 | 0.542 |
| Cross-Attention | 556,417 | 0.507 |

## EER (%) — Raw ECAPA

| noise_type | -20.0 | -15.0 | -10.0 | -5.0 | 0.0 | 5.0 | 10.0 | 15.0 | 20.0 | clean |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| clean | — | — | — | — | — | — | — | — | — | 0.90 |
| babble | 46.23 | 42.49 | 35.07 | 23.07 | 10.55 | 4.29 | 1.86 | 1.16 | 0.98 | — |
| music | 39.90 | 32.94 | 22.37 | 9.90 | 3.67 | 1.71 | 1.28 | 1.05 | 0.94 | — |
| noise | 24.36 | 16.24 | 8.71 | 3.98 | 2.17 | 1.41 | 1.16 | 1.07 | 0.99 | — |

## EER (%) — Enhancer-only

| noise_type | -20.0 | -15.0 | -10.0 | -5.0 | 0.0 | 5.0 | 10.0 | 15.0 | 20.0 | clean |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| clean | — | — | — | — | — | — | — | — | — | 0.90 |
| babble | 38.80 | 31.30 | 22.64 | 15.17 | 8.38 | 4.55 | 2.18 | 1.30 | 1.04 | — |
| music | 38.90 | 31.59 | 20.10 | 8.32 | 3.37 | 1.74 | 1.27 | 1.04 | 0.98 | — |
| noise | 24.37 | 16.14 | 8.74 | 4.29 | 2.30 | 1.45 | 1.18 | 1.10 | 1.03 | — |

## EER (%) — MLP Fusion

| noise_type | -20.0 | -15.0 | -10.0 | -5.0 | 0.0 | 5.0 | 10.0 | 15.0 | 20.0 | clean |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| clean | — | — | — | — | — | — | — | — | — | 0.96 |
| babble | 38.06 | 29.98 | 21.07 | 13.48 | 7.48 | 4.06 | 1.96 | 1.22 | 1.06 | — |
| music | 35.90 | 27.36 | 16.99 | 7.18 | 3.19 | 1.76 | 1.32 | 1.10 | 0.97 | — |
| noise | 18.59 | 11.73 | 6.59 | 3.43 | 2.07 | 1.45 | 1.24 | 1.13 | 1.03 | — |

## EER (%) — Self-Attention Fusion

| noise_type | -20.0 | -15.0 | -10.0 | -5.0 | 0.0 | 5.0 | 10.0 | 15.0 | 20.0 | clean |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| clean | — | — | — | — | — | — | — | — | — | 0.94 |
| babble | 37.72 | 29.28 | 20.61 | 13.44 | 7.51 | 4.11 | 1.97 | 1.27 | 1.04 | — |
| music | 35.82 | 26.96 | 16.82 | 7.26 | 3.13 | 1.71 | 1.28 | 1.03 | 0.98 | — |
| noise | 18.65 | 11.85 | 6.86 | 3.60 | 2.13 | 1.42 | 1.18 | 1.12 | 1.04 | — |

## EER (%) — Cross-Attention Fusion

| noise_type | -20.0 | -15.0 | -10.0 | -5.0 | 0.0 | 5.0 | 10.0 | 15.0 | 20.0 | clean |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|
| clean | — | — | — | — | — | — | — | — | — | 0.95 |
| babble | 37.88 | 29.63 | 20.91 | 13.51 | 7.53 | 4.13 | 1.99 | 1.31 | 1.07 | — |
| music | 35.98 | 27.86 | 17.18 | 7.32 | 3.25 | 1.79 | 1.29 | 1.08 | 0.97 | — |
| noise | 18.83 | 11.82 | 6.75 | 3.53 | 2.10 | 1.48 | 1.24 | 1.11 | 1.04 | — |

## Summary by noise type (avg across SNRs)

| Noise type | Raw | Enh | MLP | SA | CA |
|---|---:|---:|---:|---:|---:|
| babble | 18.41 | 13.93 | 13.15 | **12.99** | 13.11 |
| music | 12.64 | 11.92 | 10.64 | **10.56** | 10.75 |
| noise | 6.68 | 6.73 | **5.25** | 5.32 | 5.32 |
| **ALL** | 12.58 | 10.86 | 9.68 | **9.62** | 9.73 |

## Paper subset (0 to −20 dB)

| | Raw | Enh | MLP | SA | CA |
|---|---:|---:|---:|---:|---:|
| Paper subset | 21.44 | 18.29 | 16.21 | **16.11** | 16.27 |

## Clean

| | Raw | Enh | MLP | SA | CA |
|---|---:|---:|---:|---:|---:|
| Clean | **0.90** | **0.90** | 0.96 | 0.94 | 0.95 |

## Best model per region

| Region | Best model | EER (%) |
|---|---|---:|
| All noisy | SA | 9.62 |
| Paper subset | SA | 16.11 |

## Improvement over raw (Δ EER, negative = better)

| Noise type | Enh | MLP | SA | CA |
|---|---:|---:|---:|---:|
| babble | −4.48 | −5.26 | **−5.42** | −5.30 |
| music | −0.72 | −2.00 | **−2.08** | −1.89 |
| noise | +0.06 | **−1.43** | −1.36 | −1.35 |
| **ALL** | −1.71 | −2.89 | **−2.95** | −2.85 |